### 1. Import necessary packages and set problem specific data

In [1]:
# Import TensorFlow and NumPy
import tensorflow as tf
import numpy as np

# Set data type
DTYPE='float32'
tf.keras.backend.set_floatx(DTYPE)

# Set constants
m1 = 523
m2 = 1046
c1 = 13687
c2 = 9952
k1 = 45929
k2 = 40731

# Define initial condition
def fun_u_0(t):
    return 0

# Define boundary condition


# Define residual of the PDE
def fun_r_1(t, x, x_t, x_tt, y, y_t, y_tt):
    return m1*x_tt + (c1+c2)*x_t + (k1+k2)*x - c2*y_t -k2*y

def fun_r_2(t, x, x_t, x_tt, y, y_t, y_tt):
    return m2*y_tt + c2*y_t - c1*x_t + k2*y - k2*x

D:\Anaconda3\lib\site-packages\h5py\__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters


### 2. Generate a set of collocation points

We assume that the collocation points $X_r$ as well as the points for the initial time and boundary data $X_0$ and $X_b$ are generated by random sampling from a uniform distribution.

In [2]:
# Set number of data points
N_0 = 50
N_b = 50
N_r = 10000

# Set boundary
tmin = 0
tmax = 0.25

# Lower bounds
lb = tf.constant(tmin, dtype=DTYPE)
# Upper bounds
ub = tf.constant(tmax, dtype=DTYPE)

# Set random seed for reproducible results
tf.random.set_random_seed(0)

# Draw uniform sample points for initial boundary data
t_0 = tf.ones((N_0,1), dtype=DTYPE)*lb
X_0 = tf.concat(t_0, axis=0)

# Evaluate intitial condition at x_0
u_0 = tf.ones((N_0,1), dtype=DTYPE)*lb

# Boundary data


# Evaluate boundary condition at (t_b,x_b)


# Draw uniformly sampled collocation points
t_r = tf.random.uniform((N_r,1), lb, ub, dtype=DTYPE)

# Collect boundary and inital data in lists
t_data = [X_0]
u_data = [u_0]
v_data = [u_0]

### 3. Set up network architecture

In this example, adopted from 
([Raissi et al., 2017 (Part I)](https://arxiv.org/abs/1711.10561)), we assume a feedforward neural network of the following structure:
- the input is scaled elementwise to lie in the interval $[-1,1]$,
- followed by 8 fully connected layers each containing 20 neurons and each followed by a hyperbolic tangent activation function,
- one fully connected output layer.

In [8]:
def init_model(num_hidden_layers=8, num_neurons_per_layer=20):
    # Initialize a feedforward neural network
    model = tf.keras.Sequential()

    # Input is two-dimensional (time + one spatial dimension)
    model.add(tf.keras.Input(shape=(1,)))

    # Introduce a scaling layer to map input to [lb, ub]
#     scaling_layer = tf.keras.layers.Lambda(
#                 lambda x: 2.0*(x - lb)/(ub - lb) - 1.0)
#     model.add(scaling_layer)

    # Append hidden layers
    for _ in range(num_hidden_layers):
        model.add(tf.keras.layers.Dense(num_neurons_per_layer,
            activation=tf.keras.activations.get('tanh'),
            kernel_initializer='glorot_normal'))

    # Output is one-dimensional
    model.add(tf.keras.layers.Dense(1))
    
    return model

### 4. Determine loss and gradient

In [4]:
def get_r_1(model, t_r):
    # A tf.GradientTape is used to compute derivatives in TensorFlow
    with tf.GradientTape(persistent=True) as tape:
        # Split t and x to compute partial derivatives
        t = t_r

        # Variables t and x are watched during tape
        # to compute derivatives u_t and u_x
        tape.watch(t)

        # Determine residual 
        u = model(t)
        v = model(t)

        # Compute gradient u_x within the GradientTape
        # since we need second derivatives
            
        u_t = tape.gradient(u, t)
        u_tt = tape.gradient(u_t, t)
        v_t = tape.gradient(v, t)
        v_tt = tape.gradient(v_t, t)

    del tape

    return fun_r_1(t, u, u_t, u_tt, v, v_t, v_tt)

def get_r_2(model, t_r):
    # A tf.GradientTape is used to compute derivatives in TensorFlow
    with tf.GradientTape(persistent=True) as tape:
        # Split t and x to compute partial derivatives
        t = t_r

        # Variables t and x are watched during tape
        # to compute derivatives u_t and u_x
        tape.watch(t)

        # Determine residual 
        u = model(t)
        v = model(t)

        # Compute gradient u_x within the GradientTape
        # since we need second derivatives
            
        u_t = tape.gradient(u, t)
        u_tt = tape.gradient(u_t, t)
        v_t = tape.gradient(v, t)
        v_tt = tape.gradient(v_t, t)

    del tape

    return fun_r_2(t, u, u_t, u_tt, v, v_t, v_tt)

In [5]:
def compute_loss_1(model, t_r, t_data, u_data):
    # Compute phi^r
    r = get_r_1(model, t_r)
    phi_r = tf.reduce_mean(tf.square(r))

    # Initialize loss
    loss = phi_r

    # Add phi^0 and phi^b to the loss
    for i in range(len(t_data)):
        u_pred = model(t_data[i])
        loss += tf.reduce_mean(tf.square(u_data[i] - u_pred))
    
    return loss
    
def compute_loss_2(model, t_r, t_data, v_data):
    # Compute phi^r
    r = get_r_2(model, t_r)
    phi_r = tf.reduce_mean(tf.square(r))

    # Initialize loss
    loss = phi_r

    # Add phi^0 and phi^b to the loss
    for i in range(len(t_data)):
        v_pred = model(t_data[i])
        loss += tf.reduce_mean(tf.square(v_data[i] - v_pred))
    
    return loss

In [6]:
def get_grad_1(model, t_r, t_data, u_data):
    
    with tf.GradientTape(persistent=True) as tape:
        # This tape is for derivatives with
        # respect to trainable variables
        tape.watch(model.trainable_variables)
        loss = compute_loss_1(model, t_r, t_data, u_data)

    g = tape.gradient(loss, model.trainable_variables)
    del tape

    return loss, g

def get_grad_2(model, t_r, t_data, v_data):
    
    with tf.GradientTape(persistent=True) as tape:
        # This tape is for derivatives with
        # respect to trainable variables
        tape.watch(model.trainable_variables)
        loss = compute_loss_2(model, t_r, t_data, v_data)

    g = tape.gradient(loss, model.trainable_variables)
    del tape

    return loss, g

### 5. Set up optimizer and train model

In [7]:
# Initialize model aka u_\theta
model = init_model()

# We choose a piecewise decay of the learning rate, i.e., the
# step size in the gradient descent type algorithm
# the first 1000 steps use a learning rate of 0.01
# from 1000 - 3000: learning rate = 0.001
# from 3000 onwards: learning rate = 0.0005

lr = tf.keras.optimizers.schedules.PiecewiseConstantDecay([1000,3000],[1e-2,1e-3,5e-4])

# Choose the optimizer
optim = tf.keras.optimizers.Adam(learning_rate=lr)

TypeError: The added layer must be an instance of class Layer. Found: Tensor("input_1:0", shape=(?, 1), dtype=float32)

In [ ]:
from time import time

# Define one training step as a TensorFlow function to increase speed of training
def train_step_1():
    # Compute current loss and gradient w.r.t. parameters
    loss, grad_theta = get_grad_1(model, t_r, t_data, u_data)
    
    # Perform gradient descent step
    optim.apply_gradients(zip(grad_theta, model.trainable_variables))
    
    return loss

# Number of training epochs
N = 1000
hist_1 = []

# Start timer
t0 = time()

for i in range(N+1):
    
    loss_1 = train_step_1()
    
    # Append current loss to hist
    hist_1.append(loss_1.numpy())
    
    # Output current loss after 50 iterates
    if i%50 == 0:
        print('It {:05d}: loss = {:10.8e}'.format(i,loss_1))
        
# Print computation time
print('\nComputation time: {} seconds'.format(time()-t0))


import matplotlib.pyplot as plt

# Set up meshgrid
N = 600
tspace = np.linspace(lb, ub, N + 1)

# Determine predictions of u(t, x)
with tf.GradientTape(persistent=True) as tape:
  uspace= model(tf.cast(tspace,DTYPE))
  u1space = tape.gradient(model(tf.cast(tspace,DTYPE)), model.trainable_variables)
  u11space = tape.gradient(tape.gradient(model(tf.cast(tspace,DTYPE)), model.trainable_variables), model.trainable_variables)

In [ ]:
def train_step_2():
    # Compute current loss and gradient w.r.t. parameters
    loss, grad_theta = get_grad_2(model, t_r, t_data, v_data)
    
    # Perform gradient descent step
    optim.apply_gradients(zip(grad_theta, model.trainable_variables))
    
    return loss

# Number of training epochs
N = 1000
hist_2 = []

# Start timer
t0 = time()

for i in range(N+1):
    
    loss_2 = train_step_2()
    
    # Append current loss to hist
    hist_2.append(loss_2.numpy())
    
    # Output current loss after 50 iterates
    if i%50 == 0:
        print('It {:05d}: loss = {:10.8e}'.format(i,loss_2))
        
# Print computation time
print('\nComputation time: {} seconds'.format(time()-t0))

# Set up meshgrid
N = 600
tspace = np.linspace(lb, ub, N + 1)

# Determine predictions of u(t, x)
with tf.GradientTape(persistent=True) as tape:
  vspace= model(tf.cast(tspace,DTYPE))
  v1space = tape.gradient(model(tf.cast(tspace,DTYPE)), model.trainable_variables)
  v11space = tape.gradient(tape.gradient(model(tf.cast(tspace,DTYPE)), model.trainable_variables), model.trainable_variables)

### Plot solution

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot()
ax1.plot(tspace, uspace+vspace);

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot()
ax1.plot(tspace, u1space+v1space);

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot()
ax1.plot(tspace, u11space+v11space);

## Parameter identification setting

In [ ]:
class PINN_NeuralNet(tf.keras.Model):

    def __init__(self, lb, ub, 
            output_dim=1,
            num_hidden_layers=8, 
            num_neurons_per_layer=20,
            activation='tanh',
            kernel_initializer='glorot_normal',
            **kwargs):
        super().__init__(**kwargs)

        self.num_hidden_layers = num_hidden_layers
        self.output_dim = output_dim
        self.lb = lb
        self.ub = ub
        
        # Define NN architecture
        self.scale = tf.keras.layers.Lambda(
            lambda x: 2.0*(x - lb)/(ub - lb) - 1.0)
        self.hidden = [tf.keras.layers.Dense(num_neurons_per_layer,
                             activation=tf.keras.activations.get(activation),
                             kernel_initializer=kernel_initializer)
                           for _ in range(self.num_hidden_layers)]
        self.out = tf.keras.layers.Dense(output_dim)
        
    def call(self, X):
        Z = self.scale(X)
        for i in range(self.num_hidden_layers):
            Z = self.hidden[i](Z)
        return self.out(Z)

In [ ]:
import scipy.optimize

class PINNSolver():
    def __init__(self, model, X_r):
        self.model = model
        
        # Store collocation points
        self.t = X_r[:,0:1]
        self.x = X_r[:,1:2]
        
        # Initialize history of losses and global iteration counter
        self.hist = []
        self.iter = 0
    
    def get_r(self):
        
        with tf.GradientTape(persistent=True) as tape:
            # Watch variables representing t and x during this GradientTape
            tape.watch(self.t)
            tape.watch(self.x)
            
            # Compute current values u(t,x)
            u = self.model(tf.stack([self.t[:,0], self.x[:,0]], axis=1))
            
            u_x = tape.gradient(u, self.x)
            
        u_t = tape.gradient(u, self.t)
        u_xx = tape.gradient(u_x, self.x)
        
        del tape
        
        return self.fun_r(self.t, self.x, u, u_t, u_x, u_xx)
    
    def loss_fn(self, X, u):
        
        # Compute phi_r
        r = self.get_r()
        phi_r = tf.reduce_mean(tf.square(r))
        
        # Initialize loss
        loss = phi_r

        # Add phi_0 and phi_b to the loss
        for i in range(len(X)):
            u_pred = self.model(X[i])
            loss += tf.reduce_mean(tf.square(u[i] - u_pred))
        
        return loss
    
    def get_grad(self, X, u):
        with tf.GradientTape(persistent=True) as tape:
            # This tape is for derivatives with
            # respect to trainable variables
            tape.watch(self.model.trainable_variables)
            loss = self.loss_fn(X, u)
            
        g = tape.gradient(loss, self.model.trainable_variables)
        del tape
        
        return loss, g
    
    def fun_r(self, t, x, u, u_t, u_x, u_xx):
        """Residual of the PDE"""
        return u_t + u * u_x - viscosity * u_xx
    
    def solve_with_TFoptimizer(self, optimizer, X, u, N=1001):
        """This method performs a gradient descent type optimization."""
        
        @tf.function
        def train_step():
            loss, grad_theta = self.get_grad(X, u)
            
            # Perform gradient descent step
            optimizer.apply_gradients(zip(grad_theta, self.model.trainable_variables))
            return loss
        
        for i in range(N):
            
            loss = train_step()
            
            self.current_loss = loss.numpy()
            self.callback()

    def solve_with_ScipyOptimizer(self, X, u, method='L-BFGS-B', **kwargs):
        """This method provides an interface to solve the learning problem
        using a routine from scipy.optimize.minimize.
        (Tensorflow 1.xx had an interface implemented, which is not longer
        supported in Tensorflow 2.xx.)
        Type conversion is necessary since scipy-routines are written in Fortran
        which requires 64-bit floats instead of 32-bit floats."""
        
        def get_weight_tensor():
            """Function to return current variables of the model
            as 1d tensor as well as corresponding shapes as lists."""
            
            weight_list = []
            shape_list = []
            
            # Loop over all variables, i.e. weight matrices, bias vectors and unknown parameters
            for v in self.model.variables:
                shape_list.append(v.shape)
                weight_list.extend(v.numpy().flatten())
                
            weight_list = tf.convert_to_tensor(weight_list)
            return weight_list, shape_list

        x0, shape_list = get_weight_tensor()
        
        def set_weight_tensor(weight_list):
            """Function which sets list of weights
            to variables in the model."""
            idx = 0
            for v in self.model.variables:
                vs = v.shape
                
                # Weight matrices
                if len(vs) == 2:  
                    sw = vs[0]*vs[1]
                    new_val = tf.reshape(weight_list[idx:idx+sw],(vs[0],vs[1]))
                    idx += sw
                
                # Bias vectors
                elif len(vs) == 1:
                    new_val = weight_list[idx:idx+vs[0]]
                    idx += vs[0]
                    
                # Variables (in case of parameter identification setting)
                elif len(vs) == 0:
                    new_val = weight_list[idx]
                    idx += 1
                    
                # Assign variables (Casting necessary since scipy requires float64 type)
                v.assign(tf.cast(new_val, DTYPE))
        
        def get_loss_and_grad(w):
            """Function that provides current loss and gradient
            w.r.t the trainable variables as vector. This is mandatory
            for the LBFGS minimizer from scipy."""
            
            # Update weights in model
            set_weight_tensor(w)
            # Determine value of \phi and gradient w.r.t. \theta at w
            loss, grad = self.get_grad(X, u)
            
            # Store current loss for callback function            
            loss = loss.numpy().astype(np.float64)
            self.current_loss = loss            
            
            # Flatten gradient
            grad_flat = []
            for g in grad:
                grad_flat.extend(g.numpy().flatten())
            
            # Gradient list to array
            grad_flat = np.array(grad_flat,dtype=np.float64)
            
            # Return value and gradient of \phi as tuple
            return loss, grad_flat
        
        
        return scipy.optimize.minimize(fun=get_loss_and_grad,
                                       x0=x0,
                                       jac=True,
                                       method=method,
                                       callback=self.callback,
                                       **kwargs)
        
    def callback(self, xr=None):
        if self.iter % 50 == 0:
            print('It {:05d}: loss = {:10.8e}'.format(self.iter,self.current_loss))
        self.hist.append(self.current_loss)
        self.iter+=1
        
    
    def plot_solution(self, **kwargs):
        N = 600
        tspace = np.linspace(self.model.lb[0], self.model.ub[0], N+1)
        xspace = np.linspace(self.model.lb[1], self.model.ub[1], N+1)
        T, X = np.meshgrid(tspace, xspace)
        Xgrid = np.vstack([T.flatten(),X.flatten()]).T
        upred = self.model(tf.cast(Xgrid,DTYPE))
        U = upred.numpy().reshape(N+1,N+1)
        fig = plt.figure(figsize=(9,6))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_surface(T, X, U, cmap='viridis', **kwargs)
        ax.set_xlabel('$t$')
        ax.set_ylabel('$x$')
        ax.set_zlabel('$u_\\theta(t,x)$')
        ax.view_init(35,35)
        return ax
        
    def plot_loss_history(self, ax=None):
        if not ax:
            fig = plt.figure(figsize=(7,5))
            ax = fig.add_subplot(111)
        ax.semilogy(range(len(self.hist)), self.hist,'k-')
        ax.set_xlabel('$n_{epoch}$')
        ax.set_ylabel('$\\phi^{n_{epoch}}$')
        return ax

In [ ]:
class CarCrashPINNSolver(PINNSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    m1 = 523
    m2 = 1046
    c1 = 13687
    c2 = 9952
    k1 = 45929
    k2 = 40731

    def fun_r_1(self, t, x, x_t, x_tt, y, y_t, y_tt):
        return m1*x_tt + (c1+c2)*x_t + (k1+k2)*x - c2*y_t -k2*y

    def fun_r_2(self, t, x, x_t, x_tt, y, y_t, y_tt):
        return m2*y_tt + c2*y_t - c1*x_t + k2*y - k2*x
    
    def get_r_1(model, t_r):
      # A tf.GradientTape is used to compute derivatives in TensorFlow
      with tf.GradientTape(persistent=True) as tape:
          # Split t and x to compute partial derivatives
          t = t_r

          # Variables t and x are watched during tape
          # to compute derivatives u_t and u_x
          tape.watch(t)

          # Determine residual 
          u = model(t)
          v = model(t)

          # Compute gradient u_x within the GradientTape
          # since we need second derivatives
              
          u_t = tape.gradient(u, t)
          u_tt = tape.gradient(u_t, t)
          v_t = tape.gradient(v, t)
          v_tt = tape.gradient(v_t, t)

      del tape

      return fun_r_1(t, u, u_t, u_tt, v, v_t, v_tt)

      def get_r_2(model, t_r):
          # A tf.GradientTape is used to compute derivatives in TensorFlow
          with tf.GradientTape(persistent=True) as tape:
              # Split t and x to compute partial derivatives
              t = t_r

              # Variables t and x are watched during tape
              # to compute derivatives u_t and u_x
              tape.watch(t)

              # Determine residual 
              u = model(t)
              v = model(t)

              # Compute gradient u_x within the GradientTape
              # since we need second derivatives
                  
              u_t = tape.gradient(u, t)
              u_tt = tape.gradient(u_t, t)
              v_t = tape.gradient(v, t)
              v_tt = tape.gradient(v_t, t)

          del tape

          return fun_r_2(t, u, u_t, u_tt, v, v_t, v_tt)

In [ ]:
class PINNIdentificationNet(PINN_NeuralNet):
    def __init__(self, *args, **kwargs):
        
        # Call init of base class
        super().__init__(*args,**kwargs)
        
        # Initialize variable for c1, c2, k1, k2
        self.c1 = tf.Variable(1.0, trainable=True, dtype=DTYPE)
        self.c1_list = []
        self.c2 = tf.Variable(1.0, trainable=True, dtype=DTYPE)
        self.c2_list = []
        self.k1 = tf.Variable(1.0, trainable=True, dtype=DTYPE)
        self.k1_list = []
        self.k2 = tf.Variable(1.0, trainable=True, dtype=DTYPE)
        self.k2_list = []

In [ ]:
class CarCrashPINNIdentification(CarCrashPINNSolver):

    def fun_r_1(self, t, x, x_t, x_tt, y, y_t, y_tt):
        return m1*x_tt + (self.model.c1+self.model.c2)*x_t + (self.model.k1+self.model.k2)*x - self.model.c2*y_t -self.model.k2*y

    def fun_r_2(self, t, x, x_t, x_tt, y, y_t, y_tt):
        return m2*y_tt + self.model.c2*y_t - self.model.c1*x_t + self.model.k2*y - self.model.k2*x
    
    def callback(self, xr=None):
        c1 = self.model.c1.numpy()
        self.model.c1_list.append(c1)
        c2 = self.model.c2.numpy()
        self.model.c2_list.append(c2)
        k1 = self.model.k1.numpy()
        self.model.k1_list.append(k1)
        k2 = self.model.k2.numpy()
        self.model.k2_list.append(k2)
        
        if self.iter % 50 == 0:
            print('It {:05d}: loss = {:10.8e} c1 = {:10.8e} c2 = {:10.8e} k1 = {:10.8e} k2 = {:10.8e}'.format(self.iter, self.current_loss, c1, c2, k1, k2))
        
        self.hist.append(self.current_loss)
        self.iter += 1
        
    def plot_loss_and_param(self, axs=None):
        if axs:
            ax1, ax2 = axs
            self.plot_loss_history(ax1)
        else:
            ax1 = self.plot_loss_history()
            ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
            ax3 = ax1.twinx()
            ax3 = ax1.twinx()
            ax3 = ax1.twinx()
            ax3 = ax1.twinx()

        color = 'tab:blue'
        ax2.tick_params(axis='y', labelcolor=color)
        ax2.plot(range(len(self.hist)), self.model.lambd_list,'-',color=color)
        ax2.set_ylabel('$\\lambda^{n_{epoch}}$', color=color)

        ax2.tick_params(axis='y', labelcolor=color)
        ax2.plot(range(len(self.hist)), self.model.c1_list,'-',color=color)
        ax2.set_ylabel('$\\lambda^{n_{epoch}}$', color=color)

        ax2.tick_params(axis='y', labelcolor=color)
        ax2.plot(range(len(self.hist)), self.model.lambd_list,'-',color=color)
        ax2.set_ylabel('$\\lambda^{n_{epoch}}$', color=color)

        ax2.tick_params(axis='y', labelcolor=color)
        ax2.plot(range(len(self.hist)), self.model.lambd_list,'-',color=color)
        ax2.set_ylabel('$\\lambda^{n_{epoch}}$', color=color)

        ax2.tick_params(axis='y', labelcolor=color)
        ax2.plot(range(len(self.hist)), self.model.lambd_list,'-',color=color)
        ax2.set_ylabel('$\\lambda^{n_{epoch}}$', color=color)

        return (ax1,ax2)